In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load files (they are in current working directory)
df_metric = pd.read_csv("metric.csv")
df_trace = pd.read_csv("trace.csv")
df_log = pd.read_csv("log.csv")
df_error = pd.read_csv("error_logs.csv")

# Helper to create a compact summary row for a dataframe
def compact_summary(df, name_col_for_names=None, sample_name_limit=10, sample_cmdb_limit=10, sample_rows_limit=5, extra_columns=None):
    row_count = len(df)
    min_ts = int(df['timestamp'].min()) if row_count>0 else None
    max_ts = int(df['timestamp'].max()) if row_count>0 else None
    sample_cmdb = df['cmdb_id'].dropna().unique()[:sample_cmdb_limit].tolist()
    sample_cmdb_str = ", ".join(map(str, sample_cmdb))
    sample_names = []
    sample_names_str = None
    if name_col_for_names and name_col_for_names in df.columns:
        sample_names = df[name_col_for_names].dropna().unique()[:sample_name_limit].tolist()
        sample_names_str = ", ".join(map(str, sample_names))
    # sample rows: select key cols for compactness
    display_cols = ['timestamp','cmdb_id']
    if name_col_for_names:
        display_cols.append(name_col_for_names)
    if extra_columns:
        display_cols += [c for c in extra_columns if c in df.columns]
    display_cols = [c for c in display_cols if c in df.columns]
    sample_rows = df[display_cols].head(sample_rows_limit).copy()
    # Build a single-row summary DataFrame
    summary = pd.DataFrame([{
        'file': df.attrs.get('source_filename',''),
        'rows': row_count,
        'min_timestamp': min_ts,
        'max_timestamp': max_ts,
        'sample_cmdb_ids': sample_cmdb_str,
        'sample_names': sample_names_str
    }])
    return summary, sample_rows

# Attach source filenames (for clarity in summary)
df_metric.attrs['source_filename'] = 'metric.csv'
df_trace.attrs['source_filename'] = 'trace.csv'
df_log.attrs['source_filename'] = 'log.csv'
df_error.attrs['source_filename'] = 'error_logs.csv'

# Create summaries
metric_summary, metric_samples = compact_summary(df_metric, name_col_for_names='kpi_name', sample_name_limit=10, sample_cmdb_limit=10, sample_rows_limit=5, extra_columns=['value'])
trace_summary, trace_samples = compact_summary(df_trace, name_col_for_names='trace_name', sample_name_limit=10, sample_cmdb_limit=10, sample_rows_limit=5, extra_columns=['value'])
log_summary, log_samples = compact_summary(df_log, name_col_for_names='log_name', sample_name_limit=10, sample_cmdb_limit=10, sample_rows_limit=5, extra_columns=['value'])

# error_logs: a bit different: include up to 5 distinct cmdb_id and up to 5 sample messages
row_count = len(df_error)
min_ts = int(df_error['timestamp'].min()) if row_count>0 else None
max_ts = int(df_error['timestamp'].max()) if row_count>0 else None
error_cmdbs = df_error['cmdb_id'].dropna().unique()[:5].tolist()
error_cmdbs_str = ", ".join(map(str, error_cmdbs))
error_messages = df_error['message'].dropna().unique()[:5].tolist()
error_messages_str = " | ".join(map(str, error_messages))
error_summary = pd.DataFrame([{
    'file': df_error.attrs.get('source_filename','error_logs.csv'),
    'rows': row_count,
    'min_timestamp': min_ts,
    'max_timestamp': max_ts,
    'sample_cmdb_ids': error_cmdbs_str,
    'sample_messages_preview': error_messages_str
}])
error_samples = df_error[['timestamp','cmdb_id','message']].head(5).copy()

# Final compact outputs (tabular). Display summaries and sample rows for each file.
metric_summary, metric_samples, trace_summary, trace_samples, log_summary, log_samples, error_summary, error_samples

```
Out[1]:
```
summary = (
    "Metric (metric.csv): 144,252 rows; timestamps 1647748800–1647750540. "
    "Example cmdb_ids: adservice, adservice-0, adservice-1, ...; "
    "Example KPIs: app.grpc.count, app.grpc.mrt, app.grpc.rr, app.grpc.sr, app.http.count. "
    "Sample row: (1647748800, adservice, app.grpc.count, 258.0).\n\n"
    "Trace (trace.csv): 36,760 rows; timestamps 1647748800–1647750540. "
    "Example cmdb_ids: adservice-0, adservice-1, adservice-2, ...; "
    "Example trace names: trace.from_frontend-0.duration_mean, trace.from_frontend-0.duration_p95, trace.from_frontend-0.error_rate, trace.from_frontend-0.row_count. "
    "Sample row: (1647748800, adservice-0, trace.from_frontend-0.duration_mean, 0.000012).\n\n"
    "Log (log.csv): 1,812 rows; timestamps 1647748800–1647750540. "
    "Example cmdb_ids: adservice-0, adservice-1, adservice-2, cartservice-...; "
    "Example log names: log.error_count, log.row_count. "
    "Sample rows: (1647748800, adservice-0, log.error_count, 0.0), (1647748800, adservice-0, log.row_count, 1642.0).\n\n"
    "Error logs (error_logs.csv): 2,016 rows; timestamps 1647748800–1647750599. "
    "Example cmdb_ids: frontend-2, frontend-0, frontend-1. "
    "Sample messages (up to 5): 'severity: warning, message: failed to retrieve...'. "
    "Sample row: (1647748800, frontend-2, 'severity: warning, message: failed to retrieve...')."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(         file    rows  min_timestamp  max_timestamp                                    sample_cmdb_ids                                       sample_names
0  metric.csv  144252     1647748800     1647750540  adservice, adservice-0, adservice-1, adservice...  app.grpc.count, app.grpc.mrt, app.grpc.rr, app...,     timestamp    cmdb_id        kpi_name       value
0  1647748800  adservice  app.grpc.count  258.000000
1  1647748800  adservice    app.grpc.mrt    0.002338
2  1647748800  adservice     app.grpc.rr  100.000000
3  1647748800  adservice     app.grpc.sr   56.976744
4  1647748800  adservice  app.http.count   24.000000,         file   rows  min_timestamp  max_timestamp                                    sample_cmdb_ids                                       sample_names
0  trace.csv  36760     1647748800     1647750540  adservice-0, adservice-1, adservice-2, adservi...  trace.from_frontend-0.duration_mean, trace.fro...,     timestamp      cmdb_id                           trace_name      value
0  1647748800  adservice-0  trace.from_frontend-0.duration_mean   0.000012
1  1647748800  adservice-0   trace.from_frontend-0.duration_p95   0.000014
2  1647748800  adservice-0     trace.from_frontend-0.error_rate   0.000000
3  1647748800  adservice-0      trace.from_frontend-0.row_count  23.000000
4  1647748800  adservice-0  trace.from_frontend-1.duration_mean   0.000012,       file  rows  min_timestamp  max_timestamp                                    sample_cmdb_ids                    sample_names
0  log.csv  1812     1647748800     1647750540  adservice-0, adservice-1, adservice-2, cartser...  log.error_count, log.row_count,     timestamp      cmdb_id         log_name   value
0  1647748800  adservice-0  log.error_count     0.0
1  1647748800  adservice-0    log.row_count  1642.0
2  1647748800  adservice-1  log.error_count     0.0
3  1647748800  adservice-1    log.row_count   148.0
4  1647748800  adservice-2  log.error_count     0.0,              file  rows  min_timestamp  max_timestamp                     sample_cmdb_ids                            sample_messages_preview
0  error_logs.csv  2016     1647748800     1647750599  frontend-2, frontend-0, frontend-1  severity: warning, message: failed to retrieve...,     timestamp     cmdb_id                                            message
0  1647748800  frontend-2  severity: warning, message: failed to retrieve...
1  1647748801  frontend-0  severity: warning, message: failed to retrieve...
2  1647748802  frontend-1  severity: warning, message: failed to retrieve...
3  1647748802  frontend-2  severity: warning, message: failed to retrieve...
4  1647748804  frontend-1  severity: warning, message: failed to retrieve...)```
```

In [ ]:
# In[2]:

import numpy as np
import pandas as pd

# Reuse loaded DataFrames: df_metric, df_trace, df_log

# 1) metric.csv: group by (cmdb_id, kpi_name)
metric_groups = df_metric.groupby(['cmdb_id', 'kpi_name']).agg(
    p95 = ('value', lambda x: float(np.percentile(x, 95)) ),
    count = ('value', 'count'),
    min_timestamp = ('timestamp', 'min'),
    max_timestamp = ('timestamp', 'max')
).reset_index()
metric_top20 = metric_groups.sort_values('p95', ascending=False).head(20)
metric_group_count = metric_groups.shape[0]

# 2) trace.csv: group by (cmdb_id, trace_name)
trace_groups = df_trace.groupby(['cmdb_id', 'trace_name']).agg(
    p95 = ('value', lambda x: float(np.percentile(x, 95)) ),
    count = ('value', 'count'),
    min_timestamp = ('timestamp', 'min'),
    max_timestamp = ('timestamp', 'max')
).reset_index()
trace_top20 = trace_groups.sort_values('p95', ascending=False).head(20)
trace_group_count = trace_groups.shape[0]

# 3) log.csv: group by (cmdb_id, log_name)
log_groups = df_log.groupby(['cmdb_id', 'log_name']).agg(
    p95 = ('value', lambda x: float(np.percentile(x, 95)) ),
    count = ('value', 'count'),
    min_timestamp = ('timestamp', 'min'),
    max_timestamp = ('timestamp', 'max')
).reset_index()
log_top20 = log_groups.sort_values('p95', ascending=False).head(20)
log_group_count = log_groups.shape[0]

# Compact outputs (limited to top 20 rows each) and group counts
metric_top20, metric_group_count, trace_top20, trace_group_count, log_top20, log_group_count

```
Out[2]:
```
summary = (
    "Metric (metric.csv): 4,818 distinct (cmdb_id,kpi_name) groups. "
    "Top 3 by global P95: "
    "1) adservice2 / runtime.java_lang_Runtime_StartTime — p95=1.647523e+12, count=30, ts=1647748800–1647750540; "
    "2) adservice / runtime.java_lang_Runtime_StartTime — p95=1.647523e+12, count=30, ts=1647748800–1647750540; "
    "3) adservice2 / runtime.java_lang_OperatingSystem_ProcessCpuTime — p95=7.594315e+11, count=30, ts=1647748800–1647750540. "
    "Results limited to top 20 rows sorted by P95.\n\n"
    "Trace (trace.csv): 1,232 distinct (cmdb_id,trace_name) groups. "
    "Top 3 by global P95: "
    "1) frontend-2 / trace.self.row_count — p95=1393.20, count=30, ts=1647748800–1647750540; "
    "2) frontend-0 / trace.self.row_count — p95=1280.70, count=30, ts=1647748800–1647750540; "
    "3) frontend-1 / trace.self.row_count — p95=1012.90, count=30, ts=1647748800–1647750540. "
    "Results limited to top 20 rows sorted by P95.\n\n"
    "Log (log.csv): 62 distinct (cmdb_id,log_name) groups. "
    "Top 3 by global P95: "
    "1) frontend-2 / log.row_count — p95=2262.35, count=30, ts=1647748800–1647750540; "
    "2) frontend-0 / log.row_count — p95=1956.70, count=30, ts=1647748800–1647750540; "
    "3) adservice-0 / log.row_count — p95=1655.00, count=30, ts=1647748800–1647750540. "
    "Results limited to top 20 rows sorted by P95."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id                                           kpi_name           p95  count  min_timestamp  max_timestamp
829   adservice2                runtime.java_lang_Runtime_StartTime  1.647523e+12     30     1647748800     1647750540
250    adservice                runtime.java_lang_Runtime_StartTime  1.647523e+12     30     1647748800     1647750540
823   adservice2   runtime.java_lang_OperatingSystem_ProcessCpuTime  7.594315e+11     30     1647748800     1647750540
244    adservice   runtime.java_lang_OperatingSystem_ProcessCpuTime  7.417355e+11     30     1647748800     1647750540
832   adservice2   runtime.java_lang_Threading_CurrentThreadCpuTime  7.096198e+10     30     1647748800     1647750540
253    adservice   runtime.java_lang_Threading_CurrentThreadCpuTime  7.005603e+10     30     1647748800     1647750540
255    adservice  runtime.java_lang_Threading_CurrentThreadUserTime  6.813487e+10     30     1647748800     1647750540
834   adservice2  runtime.java_lang_Threading_CurrentThreadUserTime  6.797683e+10     30     1647748800     1647750540
252    adservice  runtime.java_lang_Threading_CurrentThreadAlloc...  5.544734e+10     30     1647748800     1647750540
831   adservice2  runtime.java_lang_Threading_CurrentThreadAlloc...  5.533192e+10     30     1647748800     1647750540
3255      node-4                                  system.disk.total  7.616300e+09     30     1647748800     1647750540
3259      node-4                             system.fs.inodes.total  6.711601e+09     30     1647748800     1647750540
3257      node-4                              system.fs.inodes.free  6.709018e+09     30     1647748800     1647750540
3079      node-1                                  system.disk.total  5.867011e+09     30     1647748800     1647750540
3196      node-3                                  system.disk.total  5.867011e+09     30     1647748800     1647750540
3138      node-2                                  system.disk.total  5.867011e+09     30     1647748800     1647750540
3256      node-4                                   system.disk.used  5.746759e+09     30     1647748800     1647750540
3142      node-2                             system.fs.inodes.total  5.043277e+09     30     1647748800     1647750540
3083      node-1                             system.fs.inodes.total  5.043222e+09     30     1647748800     1647750540
3200      node-3                             system.fs.inodes.total  5.042987e+09     30     1647748800     1647750540, 4818,                        cmdb_id                                   trace_name      p95  count  min_timestamp  max_timestamp
759                 frontend-2                         trace.self.row_count  1393.20     30     1647748800     1647750540
575                 frontend-0                         trace.self.row_count  1280.70     30     1647748800     1647750540
667                 frontend-1                         trace.self.row_count  1012.90     30     1647748800     1647750540
851                frontend2-0                         trace.self.row_count   747.75     30     1647748800     1647750540
871                frontend2-0  trace.to_productcatalogservice2-0.row_count   352.90     30     1647748800     1647750540
1047  productcatalogservice2-0             trace.from_frontend2-0.row_count   352.90     30     1647748800     1647750540
527         currencyservice2-0             trace.from_frontend2-0.row_count   243.50     30     1647748800     1647750540
867                frontend2-0        trace.to_currencyservice2-0.row_count   243.50     30     1647748800     1647750540
815                 frontend-2   trace.to_productcatalogservice-1.row_count   224.20     30     1647748800     1647750540
811                 frontend-2   trace.to_productcatalogservice-0.row_count   224.20     30     1647748800     1647750540
983    productcatalogservice-1              trace.from_frontend-2.row_count   224.20     30     1647748800     1647750540
1023   productcatalogservice-2              trace.from_frontend-2.row_count   224.20     30     1647748800     1647750540
819                 frontend-2   trace.to_productcatalogservice-2.row_count   224.20     30     1647748800     1647750540
943    productcatalogservice-0              trace.from_frontend-2.row_count   224.20     30     1647748800     1647750540
1015   productcatalogservice-2              trace.from_frontend-0.row_count   198.10     30     1647748800     1647750540
975    productcatalogservice-1              trace.from_frontend-0.row_count   198.10     30     1647748800     1647750540
631                 frontend-0   trace.to_productcatalogservice-1.row_count   198.10     30     1647748800     1647750540
635                 frontend-0   trace.to_productcatalogservice-2.row_count   198.10     30     1647748800     1647750540
627                 frontend-0   trace.to_productcatalogservice-0.row_count   197.65     30     1647748800     1647750540
935    productcatalogservice-0              trace.from_frontend-0.row_count   197.65     30     1647748800     1647750540, 1232,                     cmdb_id       log_name      p95  count  min_timestamp  max_timestamp
35               frontend-2  log.row_count  2262.35     30     1647748800     1647750540
31               frontend-0  log.row_count  1956.70     30     1647748800     1647750540
1               adservice-0  log.row_count  1655.00     30     1647748800     1647750540
33               frontend-1  log.row_count  1527.35     30     1647748800     1647750540
43  productcatalogservice-0  log.row_count  1307.40     30     1647748800     1647750540
45  productcatalogservice-1  log.row_count  1273.75     30     1647748800     1647750540
47  productcatalogservice-2  log.row_count  1271.90     30     1647748800     1647750540
7             cartservice-0  log.row_count  1149.75     30     1647748800     1647750540
11            cartservice-2  log.row_count  1148.90     30     1647748800     1647750540
9             cartservice-1  log.row_count  1146.40     30     1647748800     1647750540
19        currencyservice-0  log.row_count  1023.05     30     1647748800     1647750540
21        currencyservice-1  log.row_count  1017.50     30     1647748800     1647750540
23        currencyservice-2  log.row_count  1014.95     30     1647748800     1647750540
49  recommendationservice-0  log.row_count   272.40     30     1647748800     1647750540
53  recommendationservice-2  log.row_count   271.85     30     1647748800     1647750540
51  recommendationservice-1  log.row_count   271.40     30     1647748800     1647750540
3               adservice-1  log.row_count   151.10     30     1647748800     1647750540
5               adservice-2  log.row_count   151.10     30     1647748800     1647750540
59        shippingservice-1  log.row_count   144.55     30     1647748800     1647750540
61        shippingservice-2  log.row_count   143.65     30     1647748800     1647750540, 62)```
```

In [ ]:
# In[3]:

import numpy as np
import pandas as pd
from datetime import datetime
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing DataFrames: df_metric, df_trace, df_log

# 1) Compute global p95 and p05 for each group (before any filtering)
metric_stats = df_metric.groupby(['cmdb_id','kpi_name'])['value'].agg(
    global_p95=lambda x: float(np.percentile(x,95)),
    global_p05=lambda x: float(np.percentile(x,5)),
    group_count='count'
).reset_index()

trace_stats = df_trace.groupby(['cmdb_id','trace_name'])['value'].agg(
    global_p95=lambda x: float(np.percentile(x,95)),
    global_p05=lambda x: float(np.percentile(x,5)),
    group_count='count'
).reset_index().rename(columns={'trace_name':'kpi_name'})

log_stats = df_log.groupby(['cmdb_id','log_name'])['value'].agg(
    global_p95=lambda x: float(np.percentile(x,95)),
    global_p05=lambda x: float(np.percentile(x,5)),
    group_count='count'
).reset_index().rename(columns={'log_name':'kpi_name'})

# 2) Incident window (UTC+8) -> epoch seconds
start_local = tz.localize(datetime.strptime("2022-03-20 12:00:00", "%Y-%m-%d %H:%M:%S"))
end_local   = tz.localize(datetime.strptime("2022-03-20 12:30:00", "%Y-%m-%d %H:%M:%S"))
# convert to POSIX epoch seconds (UTC)
start_ts = int(start_local.astimezone(pytz.UTC).timestamp())
end_ts   = int(end_local.astimezone(pytz.UTC).timestamp())

# Filter rows within the window (timestamps in data are epoch seconds)
df_metric_win = df_metric[(df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)].copy()
df_trace_win  = df_trace[(df_trace['timestamp']  >= start_ts) & (df_trace['timestamp']  <= end_ts)].copy()
df_log_win    = df_log[(df_log['timestamp']    >= start_ts) & (df_log['timestamp']    <= end_ts)].copy()

# Helper to detect anomalies per (cmdb_id, kpi_name) group for a windowed dataframe and precomputed stats
def detect_anomalies(window_df, stats_df, file_label, name_col):
    # normalize column names: ensure window_df has 'kpi_name' column
    window = window_df.rename(columns={name_col: 'kpi_name'}).copy()
    stats = stats_df.copy()
    # join thresholds into window rows
    merged = window.merge(stats, on=['cmdb_id','kpi_name'], how='left')
    # For each group, compute anomalies of type 'high' and 'low' if present
    groups = []
    grouped = merged.groupby(['cmdb_id','kpi_name'])
    for (cmdb, kpi), g in grouped:
        gp95 = g['global_p95'].iloc[0]
        gp05 = g['global_p05'].iloc[0]
        # high anomalies
        high_rows = g[g['value'] >= gp95]
        if not high_rows.empty:
            extreme_val = float(high_rows['value'].max())
            earliest_ts = int(high_rows['timestamp'].min())
            cnt = int(len(high_rows))
            severity = np.nan
            if gp95 != 0:
                severity = abs(extreme_val - gp95) / abs(gp95)
            groups.append({
                'file': file_label,
                'cmdb_id': cmdb,
                'kpi_name': kpi,
                'anomaly_type': 'high',
                'earliest_anomaly_timestamp': earliest_ts,
                'anomaly_count_in_window': cnt,
                'extreme_value_in_window': extreme_val,
                'global_p95': float(gp95),
                'global_p05': float(gp05),
                'severity_ratio': float(severity) if not np.isnan(severity) else np.nan
            })
        # low anomalies
        low_rows = g[g['value'] <= gp05]
        if not low_rows.empty:
            extreme_val = float(low_rows['value'].min())
            earliest_ts = int(low_rows['timestamp'].min())
            cnt = int(len(low_rows))
            severity = np.nan
            if gp05 != 0:
                severity = abs(extreme_val - gp05) / abs(gp05)
            groups.append({
                'file': file_label,
                'cmdb_id': cmdb,
                'kpi_name': kpi,
                'anomaly_type': 'low',
                'earliest_anomaly_timestamp': earliest_ts,
                'anomaly_count_in_window': cnt,
                'extreme_value_in_window': extreme_val,
                'global_p95': float(gp95),
                'global_p05': float(gp05),
                'severity_ratio': float(severity) if not np.isnan(severity) else np.nan
            })
    if groups:
        return pd.DataFrame(groups)
    else:
        return pd.DataFrame(columns=[
            'file','cmdb_id','kpi_name','anomaly_type','earliest_anomaly_timestamp',
            'anomaly_count_in_window','extreme_value_in_window','global_p95','global_p05','severity_ratio'
        ])

# Detect anomalies for each file
anomalies_metric = detect_anomalies(df_metric_win, metric_stats, 'metric.csv', 'kpi_name')
anomalies_trace  = detect_anomalies(df_trace_win, trace_stats,  'trace.csv',  'trace_name')
anomalies_log    = detect_anomalies(df_log_win,   log_stats,    'log.csv',    'log_name')

# Counts of distinct groups that had anomalies in the window per file
metric_count = anomalies_metric[['cmdb_id','kpi_name']].drop_duplicates().shape[0]
trace_count  = anomalies_trace[['cmdb_id','kpi_name']].drop_duplicates().shape[0]
log_count    = anomalies_log[['cmdb_id','kpi_name']].drop_duplicates().shape[0]

anomaly_counts = pd.DataFrame([{
    'metric_count': metric_count,
    'trace_count': trace_count,
    'log_count': log_count
}])

# Combine all anomalies, sort by severity_ratio desc, limit to top 20
combined = pd.concat([anomalies_metric, anomalies_trace, anomalies_log], ignore_index=True)
# Replace possible infinite or nan severity with -inf so they sort to bottom; but keep original values
combined['severity_sort'] = combined['severity_ratio'].replace([np.inf, -np.inf], np.nan)
combined_sorted = combined.sort_values('severity_sort', ascending=False).drop(columns=['severity_sort'])
anomalies_top20 = combined_sorted.head(20).reset_index(drop=True)

# Keep output compact: show counts and the top anomalies with selected columns
anomaly_counts, anomalies_top20[[
    'file','cmdb_id','kpi_name','anomaly_type','earliest_anomaly_timestamp',
    'anomaly_count_in_window','extreme_value_in_window','global_p95','global_p05','severity_ratio'
]].head(20)

```
Out[3]:
```
summary = (
    "Incident window: 2022-03-20 12:00:00+08 to 2022-03-20 12:30:00+08 (timestamps in outputs are epoch seconds).\n\n"
    "Groups with anomalies in the window: metric.csv=4818, trace.csv=1232, log.csv=62.\n\n"
    "Top anomalies by severity_ratio (compact):\n"
    "1) trace.csv | frontend-1 | trace.to_cartservice-0.duration_mean | high | earliest=1647748920 | count=4 | extreme=0.003000 | global_p95=0.000038 | global_p05=0.000000 | severity=77.95\n"
    "2) trace.csv | cartservice-0 | trace.from_frontend-1.duration_mean | high | earliest=1647748920 | count=4 | extreme=0.003000 | global_p95=0.000038 | global_p05=0.000000 | severity=77.95\n"
    "3) trace.csv | cartservice-1 | trace.self.duration_mean | high | earliest=1647749580 | count=2 | extreme=0.000524 | global_p95=0.000014 | global_p05=0.000000 | severity=35.14\n"
    "4) metric.csv | currencyservice-2 | container.node-5.container_memory_mapped_file | high | earliest=1647749520 | count=3 | extreme=4,075,520.0 | global_p95=151,552.0 | global_p05=8,192.0 | severity=25.89\n"
    "5) trace.csv | cartservice-2 | trace.from_frontend-1.duration_mean | high | earliest=1647749220 | count=2 | extreme=0.001458 | global_p95=0.000059 | global_p05=0.000000 | severity=23.84\n"
    "6) trace.csv | frontend-1 | trace.to_cartservice-2.duration_mean | high | earliest=1647749220 | count=2 | extreme=0.001458 | global_p95=0.000059 | global_p05=0.000000 | severity=23.84\n"
    "7) trace.csv | cartservice-2 | trace.self.duration_mean | high | earliest=1647750300 | count=2 | extreme=0.000312 | global_p95=0.000013 | global_p05=0.000000 | severity=23.09\n"
    "8) metric.csv | currencyservice-2 | container.node-5.container_fs_writes./dev/vda | high | earliest=1647749220 | count=3 | extreme=35.0 | global_p95=1.5 | global_p05=0.0 | severity=22.33\n"
    "9) trace.csv | cartservice-1 | trace.from_frontend-1.duration_mean | high | earliest=1647749580 | count=2 | extreme=0.002333 | global_p95=0.000105 | global_p05=0.000000 | severity=21.19\n"
    "10) trace.csv | frontend-1 | trace.to_cartservice-1.duration_mean | high | earliest=1647749580 | count=2 | extreme=0.002333 | global_p95=0.000105 | global_p05=0.000000 | severity=21.19\n\n"
    "Notes: severity_ratio = |extreme - threshold|/threshold using global thresholds computed from the full series (P95 or P05). "
    "Most top anomalies are 'high' short bursts in trace durations/row_counts and some container-level metrics (memory, fs writes, network)."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(   metric_count  trace_count  log_count
0          4818         1232         62,           file            cmdb_id                                           kpi_name anomaly_type  earliest_anomaly_timestamp  anomaly_count_in_window  extreme_value_in_window     global_p95   global_p05  severity_ratio
0    trace.csv         frontend-1               trace.to_cartservice-0.duration_mean         high                  1647748920                        4             3.000000e-03       0.000038     0.000000       77.947368
1    trace.csv      cartservice-0                trace.from_frontend-1.duration_mean         high                  1647748920                        4             3.000000e-03       0.000038     0.000000       77.947368
2    trace.csv      cartservice-1                           trace.self.duration_mean         high                  1647749580                        2             5.240000e-04       0.000014     0.000000       35.137931
3   metric.csv  currencyservice-2      container.node-5.container_memory_mapped_file         high                  1647749520                        3             4.075520e+06  151552.000000  8192.000000       25.891892
4    trace.csv      cartservice-2                trace.from_frontend-1.duration_mean         high                  1647749220                        2             1.458000e-03       0.000059     0.000000       23.838160
5    trace.csv         frontend-1               trace.to_cartservice-2.duration_mean         high                  1647749220                        2             1.458000e-03       0.000059     0.000000       23.838160
6    trace.csv      cartservice-2                           trace.self.duration_mean         high                  1647750300                        2             3.120000e-04       0.000013     0.000000       23.092664
7   metric.csv  currencyservice-2      container.node-5.container_fs_writes./dev/vda         high                  1647749220                        3             3.500000e+01       1.500000     0.000000       22.333333
8    trace.csv      cartservice-1                trace.from_frontend-1.duration_mean         high                  1647749580                        2             2.333000e-03       0.000105     0.000000       21.187351
9    trace.csv         frontend-1               trace.to_cartservice-1.duration_mean         high                  1647749580                        2             2.333000e-03       0.000105     0.000000       21.187351
10  metric.csv   paymentservice-0  container.node-5.container_network_receive_MB....         high                  1647749760                        2             5.578146e-01       0.025382     0.018520       20.976949
11  metric.csv    emailservice2-0  container.node-5.container_network_receive_MB....         high                  1647749820                        2             5.560107e-01       0.027254     0.020424       19.401187
12  metric.csv     emailservice-1  container.node-5.container_network_receive_MB....         high                  1647750300                        2             5.550766e-01       0.030027     0.022281       17.485647
13  metric.csv     emailservice-2  container.node-5.container_network_receive_MB....         high                  1647749880                        2             5.497174e-01       0.029758     0.020172       17.472692
14   trace.csv         frontend-1           trace.to_currencyservice-0.duration_mean         high                  1647749220                        2             1.923000e-03       0.000109     0.000086       16.609890
15   trace.csv  currencyservice-0                trace.from_frontend-1.duration_mean         high                  1647749220                        2             1.923000e-03       0.000109     0.000086       16.609890
16  metric.csv             node-1                        system.net.tcp.retrans_segs         high                  1647749760                        2             3.215000e+01       1.852000     0.434500       16.359611
17  metric.csv  currencyservice-2   container.node-5.container_fs_writes_MB./dev/vda         high                  1647749220                        3             2.304688e-01       0.013672     0.000000       15.857143
18  metric.csv  shippingservice-0  container.node-5.container_network_receive_MB....         high                  1647749880                        2             5.654950e-01       0.045476     0.036855       11.435117
19  metric.csv      redis-cart2-0  container.node-5.container_network_receive_MB....         high                  1647750480                        2             5.614557e-01       0.045775     0.033251       11.265686)```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz
from datetime import datetime

tz = pytz.timezone('Asia/Shanghai')

# Reuse df_trace and df_trace_win if available; if not, load and filter
try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")

# Incident window (UTC+8) -> epoch seconds
start_local = tz.localize(datetime.strptime("2022-03-20 12:00:00", "%Y-%m-%d %H:%M:%S"))
end_local   = tz.localize(datetime.strptime("2022-03-20 12:30:00", "%Y-%m-%d %H:%M:%S"))
start_ts = int(start_local.astimezone(pytz.UTC).timestamp())
end_ts   = int(end_local.astimezone(pytz.UTC).timestamp())

# Filter trace rows within the window (reuse df_trace_win if present)
if 'df_trace_win' not in globals():
    df_trace_win = df_trace[(df_trace['timestamp'] >= start_ts) & (df_trace['timestamp'] <= end_ts)].copy()

# Define target components and allowed trace_name sets per component
cart_cmdbs = ['cartservice-0','cartservice-1','cartservice-2']
frontend_cmdb = 'frontend-1'

allowed_cart_names = {
    'trace.to_cartservice-0.duration_mean',
    'trace.to_cartservice-1.duration_mean',
    'trace.to_cartservice-2.duration_mean',
    'trace.from_frontend-1.duration_mean',
    'trace.self.duration_mean'
}
allowed_frontend1_names = {
    'trace.self.duration_mean',
    'trace.to_cartservice-0.duration_mean',
    'trace.to_cartservice-1.duration_mean',
    'trace.to_cartservice-2.duration_mean'
}

# Filter rows by cmdb_id and trace_name according to rules
f_cart = df_trace_win[
    (df_trace_win['cmdb_id'].isin(cart_cmdbs)) &
    (df_trace_win['trace_name'].isin(allowed_cart_names))
].copy()

f_frontend1 = df_trace_win[
    (df_trace_win['cmdb_id'] == frontend_cmdb) &
    (df_trace_win['trace_name'].isin(allowed_frontend1_names))
].copy()

filtered = pd.concat([f_cart, f_frontend1], ignore_index=True)

# Group and aggregate per (cmdb_id, trace_name)
grouped = filtered.groupby(['cmdb_id','trace_name']).agg(
    count_in_window = ('value','count'),
    earliest_timestamp = ('timestamp','min'),
    latest_timestamp = ('timestamp','max'),
    max_value_in_window = ('value','max'),
    mean_value_in_window = ('value','mean')
).reset_index()

# Sort by max_value_in_window desc and limit to top 20
groups_top20 = grouped.sort_values('max_value_in_window', ascending=False).head(20).reset_index(drop=True)

# Prepare up to 10 sample rows (timestamp, cmdb_id, trace_name, value) from the filtered window
sample_rows = filtered[['timestamp','cmdb_id','trace_name','value']].head(min(10, len(filtered))).reset_index(drop=True)

# Compact outputs
groups_top20, sample_rows

```
Out[4]:
```
Summary of trace data in the incident window (2022-03-20 12:00:00+08 to 12:30:00+08), restricted to the specified trace names and components:

Groups returned (sorted by max_value_in_window, top 10 shown — total groups = 10):

1. frontend-1 | trace.self.duration_mean
   - count_in_window: 30
   - earliest_timestamp: 1647748800
   - latest_timestamp: 1647750540
   - max_value_in_window: 0.004876
   - mean_value_in_window: 0.003976

2. cartservice-0 | trace.from_frontend-1.duration_mean
   - count: 30, earliest: 1647748800, latest: 1647750540
   - max: 0.003000, mean: 0.000108

3. frontend-1 | trace.to_cartservice-0.duration_mean
   - count: 30, earliest: 1647748800, latest: 1647750540
   - max: 0.003000, mean: 0.000108

4. cartservice-1 | trace.from_frontend-1.duration_mean
   - count: 30, earliest: 1647748800, latest: 1647750540
   - max: 0.002333, mean: 0.000103

5. frontend-1 | trace.to_cartservice-1.duration_mean
   - count: 30, earliest: 1647748800, latest: 1647750540
   - max: 0.002333, mean: 0.000103

6. cartservice-2 | trace.from_frontend-1.duration_mean
   - count: 30, earliest: 1647748800, latest: 1647750540
   - max: 0.001458, mean: 0.000065

7. frontend-1 | trace.to_cartservice-2.duration_mean
   - count: 30, earliest: 1647748800, latest: 1647750540
   - max: 0.001458, mean: 0.000065

8. cartservice-0 | trace.self.duration_mean
   - count: 30, earliest: 1647748800, latest: 1647750540
   - max: 0.000679, mean: 0.000023

9. cartservice-1 | trace.self.duration_mean
   - count: 30, earliest: 1647748800, latest: 1647750540
   - max: 0.000524, mean: 0.000018

10. cartservice-2 | trace.self.duration_mean
    - count: 30, earliest: 1647748800, latest: 1647750540
    - max: 0.000312, mean: 0.000011

(These are the top 10 groups; the query returned 10 matching groups in total.)

Sample rows from the window (up to 10 rows):
- (1647748800, cartservice-0, trace.from_frontend-1.duration_mean, 0.000000)
- (1647748800, cartservice-0, trace.self.duration_mean, 0.000000)
- (1647748800, cartservice-1, trace.from_frontend-1.duration_mean, 0.000000)
- (1647748800, cartservice-1, trace.self.duration_mean, 0.000000)
- (1647748800, cartservice-2, trace.from_frontend-1.duration_mean, 0.000000)
- (1647748800, cartservice-2, trace.self.duration_mean, 0.000000)
- (1647748860, cartservice-0, trace.from_frontend-1.duration_mean, 0.000033)
- (1647748860, cartservice-0, trace.self.duration_mean, 0.000000)
- (1647748860, cartservice-1, trace.from_frontend-1.duration_mean, 0.000032)
- (1647748860, cartservice-1, trace.self.duration_mean, 0.000000)

Notes:
- All groups have full coverage in the window (count = 30), timestamps span 1647748800–1647750540.
- The largest observed durations occur for frontend-1 trace.self.duration_mean (max 0.004876) and several to/from-fronted traces with short bursts up to 0.003.

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id                            trace_name  count_in_window  earliest_timestamp  latest_timestamp  max_value_in_window  mean_value_in_window
0     frontend-1              trace.self.duration_mean               30          1647748800        1647750540             0.004876              0.003976
1  cartservice-0   trace.from_frontend-1.duration_mean               30          1647748800        1647750540             0.003000              0.000108
2     frontend-1  trace.to_cartservice-0.duration_mean               30          1647748800        1647750540             0.003000              0.000108
3  cartservice-1   trace.from_frontend-1.duration_mean               30          1647748800        1647750540             0.002333              0.000103
4     frontend-1  trace.to_cartservice-1.duration_mean               30          1647748800        1647750540             0.002333              0.000103
5  cartservice-2   trace.from_frontend-1.duration_mean               30          1647748800        1647750540             0.001458              0.000065
6     frontend-1  trace.to_cartservice-2.duration_mean               30          1647748800        1647750540             0.001458              0.000065
7  cartservice-0              trace.self.duration_mean               30          1647748800        1647750540             0.000679              0.000023
8  cartservice-1              trace.self.duration_mean               30          1647748800        1647750540             0.000524              0.000018
9  cartservice-2              trace.self.duration_mean               30          1647748800        1647750540             0.000312              0.000011,     timestamp        cmdb_id                           trace_name     value
0  1647748800  cartservice-0  trace.from_frontend-1.duration_mean  0.000000
1  1647748800  cartservice-0             trace.self.duration_mean  0.000000
2  1647748800  cartservice-1  trace.from_frontend-1.duration_mean  0.000000
3  1647748800  cartservice-1             trace.self.duration_mean  0.000000
4  1647748800  cartservice-2  trace.from_frontend-1.duration_mean  0.000000
5  1647748800  cartservice-2             trace.self.duration_mean  0.000000
6  1647748860  cartservice-0  trace.from_frontend-1.duration_mean  0.000033
7  1647748860  cartservice-0             trace.self.duration_mean  0.000000
8  1647748860  cartservice-1  trace.from_frontend-1.duration_mean  0.000032
9  1647748860  cartservice-1             trace.self.duration_mean  0.000000)```
```

In [ ]:
# In[5]:

import numpy as np
import pandas as pd
from datetime import datetime
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing variables if present: df_metric, metric_stats, start_ts, end_ts
# If any are missing, recreate them (metric_stats must be computed from full series before filtering)
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")

try:
    metric_stats
except NameError:
    metric_stats = df_metric.groupby(['cmdb_id','kpi_name'])['value'].agg(
        global_p95=lambda x: float(np.percentile(x,95)),
        global_p05=lambda x: float(np.percentile(x,5)),
        group_count='count'
    ).reset_index()

# Ensure incident window epoch seconds (UTC+8)
start_local = tz.localize(datetime.strptime("2022-03-20 12:00:00", "%Y-%m-%d %H:%M:%S"))
end_local   = tz.localize(datetime.strptime("2022-03-20 12:30:00", "%Y-%m-%d %H:%M:%S"))
start_ts = int(start_local.astimezone(pytz.UTC).timestamp())
end_ts   = int(end_local.astimezone(pytz.UTC).timestamp())

# 1) Filter metric rows in window and kpi_name containing "container.node-5"
mask_window = (df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)
mask_kpi = df_metric['kpi_name'].str.contains("container.node-5", na=False)
df_metric_filtered = df_metric[mask_window & mask_kpi].copy()

# 2) Merge global P95 into the filtered window rows (thresholds computed from full series)
df_merged = df_metric_filtered.merge(
    metric_stats[['cmdb_id','kpi_name','global_p95']],
    on=['cmdb_id','kpi_name'],
    how='left'
)

# 3) For each (cmdb_id, kpi_name) group compute required anomaly metrics
def compute_group_stats(g):
    gp95 = float(g['global_p95'].iloc[0]) if not g['global_p95'].isnull().all() else np.nan
    # anomalies where value >= global_p95
    if np.isnan(gp95):
        anomaly_rows = g.iloc[0:0]  # empty
    else:
        anomaly_rows = g[g['value'] >= gp95]
    anomaly_count = int(anomaly_rows.shape[0])
    earliest_anom_ts = int(anomaly_rows['timestamp'].min()) if anomaly_count>0 else np.nan
    max_value = float(g['value'].max()) if not g['value'].empty else np.nan
    # severity ratio: (max_value - gp95)/gp95 ; guard division by zero or nan
    if np.isnan(gp95) or gp95 == 0:
        severity = np.nan
    else:
        severity = float((max_value - gp95) / gp95)
    return pd.Series({
        'anomaly_count_in_window': anomaly_count,
        'earliest_anomaly_timestamp': earliest_anom_ts,
        'max_value_in_window': max_value,
        'global_p95': gp95,
        'severity_ratio': severity
    })

grouped_stats = df_merged.groupby(['cmdb_id','kpi_name']).apply(compute_group_stats).reset_index()

# 4) Keep only groups that had at least one anomaly (anomaly_count_in_window > 0)
anomalous_groups = grouped_stats[grouped_stats['anomaly_count_in_window'] > 0].copy()

# 5) Total number of distinct cmdb_id affected
distinct_cmdb_affected = anomalous_groups['cmdb_id'].nunique()

# 6) Sort by severity_ratio desc and limit to top 20, select required columns
top20 = anomalous_groups.sort_values('severity_ratio', ascending=False).head(20)[[
    'cmdb_id','kpi_name','anomaly_count_in_window','earliest_anomaly_timestamp',
    'max_value_in_window','global_p95','severity_ratio'
]].reset_index(drop=True)

# Compact outputs: number of affected cmdb_id and the top20 table
distinct_cmdb_affected, top20

```
Out[5]:
```
Total distinct cmdb_id affected: 35

Top anomalous (cmdb_id, kpi_name) groups (sorted by severity_ratio, up to 20 rows):

cmdb_id, kpi_name, anomaly_count_in_window, earliest_anomaly_timestamp, max_value_in_window, global_p95, severity_ratio
1) currencyservice-2, container.node-5.container_memory_mapped_file, 3, 1647750000, 4075520.0, 151552.0, 25.891892
2) currencyservice-2, container.node-5.container_fs_writes./dev/vda, 3, 1647749000, 35.0, 1.5, 22.333333
3) paymentservice-0, container.node-5.container_network_receive_MB...., 2, 1647750000, 0.5578146, 0.025382, 20.976949
4) emailservice2-0, container.node-5.container_network_receive_MB...., 2, 1647750000, 0.5560107, 0.027254, 19.401187
5) emailservice-1, container.node-5.container_network_receive_MB...., 2, 1647750300, 0.5550766, 0.030027, 17.485647
6) emailservice-2, container.node-5.container_network_receive_MB...., 2, 1647749880, 0.5497174, 0.029758, 17.472692
7) currencyservice-2, container.node-5.container_fs_writes_MB./dev/vda, 3, 1647749000, 0.2304688, 0.013672, 15.857143
8) shippingservice-0, container.node-5.container_network_receive_MB...., 2, 1647750000, 0.5654950, 0.045476, 11.435117
9) redis-cart2-0, container.node-5.container_network_receive_MB...., 2, 1647750480, 0.5614557, 0.045775, 11.265686
10) shippingservice-2, container.node-5.container_network_receive_MB...., 3, 1647749000, 0.5630975, 0.047211, 10.927209
11) checkoutservice-1, container.node-5.container_network_receive_MB...., 2, 1647750000, 0.5779099, 0.064026, 8.026159
12) checkoutservice-2, container.node-5.container_network_receive_MB...., 2, 1647750000, 0.5716677, 0.065363, 7.746034
13) adservice-1, container.node-5.container_network_receive_MB...., 2, 1647750000, 0.6014690, 0.073846, 7.144875
14) adservice-2, container.node-5.container_network_receive_MB...., 2, 1647749000, 0.6157117, 0.079463, 6.748456
15) cartservice-2, container.node-5.container_network_receive_MB...., 2, 1647749000, 0.6190906, 0.100057, 5.187358
16) currencyservice-0, container.node-5.container_fs_writes_MB./dev/vda, 2, 1647749000, 0.0859375, 0.015039, 4.714286
17) checkoutservice2-0, container.node-5.container_network_receive_MB...., 2, 1647750000, 0.2908988, 0.051304, 4.670144
18) currencyservice-0, container.node-5.container_fs_writes./dev/vda, 2, 1647749000, 9.0, 1.65, 4.454545
19) currencyservice-1, container.node-5.container_memory_mapped_file, 8, 1647749000, 1824768.0, 339968.0, 4.367470
20) adservice-1, container.node-5.container_memory_failures.con..., 2, 1647749000, 211.0, 53.425, 2.949462

Notes:
- Anomaly detection used each group's global P95 computed from the full series; anomalies are rows in the incident window with value >= that global P95.
- earliest_anomaly_timestamp are epoch seconds (UTC); all reported groups had at least one anomaly in the 12:00–12:30+08 window.

The original code execution output of IPython Kernel is also provided below for reference:

(35,                cmdb_id                                           kpi_name  anomaly_count_in_window  earliest_anomaly_timestamp  max_value_in_window     global_p95  severity_ratio
0    currencyservice-2      container.node-5.container_memory_mapped_file                      3.0                1.647750e+09         4.075520e+06  151552.000000       25.891892
1    currencyservice-2      container.node-5.container_fs_writes./dev/vda                      3.0                1.647749e+09         3.500000e+01       1.500000       22.333333
2     paymentservice-0  container.node-5.container_network_receive_MB....                      2.0                1.647750e+09         5.578146e-01       0.025382       20.976949
3      emailservice2-0  container.node-5.container_network_receive_MB....                      2.0                1.647750e+09         5.560107e-01       0.027254       19.401187
4       emailservice-1  container.node-5.container_network_receive_MB....                      2.0                1.647750e+09         5.550766e-01       0.030027       17.485647
5       emailservice-2  container.node-5.container_network_receive_MB....                      2.0                1.647750e+09         5.497174e-01       0.029758       17.472692
6    currencyservice-2   container.node-5.container_fs_writes_MB./dev/vda                      3.0                1.647749e+09         2.304688e-01       0.013672       15.857143
7    shippingservice-0  container.node-5.container_network_receive_MB....                      2.0                1.647750e+09         5.654950e-01       0.045476       11.435117
8        redis-cart2-0  container.node-5.container_network_receive_MB....                      2.0                1.647750e+09         5.614557e-01       0.045775       11.265686
9    shippingservice-2  container.node-5.container_network_receive_MB....                      3.0                1.647749e+09         5.630975e-01       0.047211       10.927209
10   checkoutservice-1  container.node-5.container_network_receive_MB....                      2.0                1.647750e+09         5.779099e-01       0.064026        8.026159
11   checkoutservice-2  container.node-5.container_network_receive_MB....                      2.0                1.647750e+09         5.716677e-01       0.065363        7.746034
12         adservice-1  container.node-5.container_network_receive_MB....                      2.0                1.647750e+09         6.014690e-01       0.073846        7.144875
13         adservice-2  container.node-5.container_network_receive_MB....                      2.0                1.647749e+09         6.157117e-01       0.079463        6.748456
14       cartservice-2  container.node-5.container_network_receive_MB....                      2.0                1.647749e+09         6.190906e-01       0.100057        5.187358
15   currencyservice-0   container.node-5.container_fs_writes_MB./dev/vda                      2.0                1.647749e+09         8.593750e-02       0.015039        4.714286
16  checkoutservice2-0  container.node-5.container_network_receive_MB....                      2.0                1.647750e+09         2.908988e-01       0.051304        4.670144
17   currencyservice-0      container.node-5.container_fs_writes./dev/vda                      2.0                1.647749e+09         9.000000e+00       1.650000        4.454545
18   currencyservice-1      container.node-5.container_memory_mapped_file                      8.0                1.647749e+09         1.824768e+06  339968.000000        4.367470
19         adservice-1  container.node-5.container_memory_failures.con...                      2.0                1.647749e+09         2.110000e+02      53.425000        2.949462)```
```

In [ ]:
# In[6]:

import numpy as np
import pandas as pd
from datetime import datetime
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing DataFrames/stats if available, otherwise load/compute
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")
try:
    metric_stats
except NameError:
    metric_stats = df_metric.groupby(['cmdb_id','kpi_name'])['value'].agg(
        global_p95=lambda x: float(np.percentile(x,95)),
        global_p05=lambda x: float(np.percentile(x,5)),
        group_count='count'
    ).reset_index()

try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")
try:
    trace_stats
except NameError:
    trace_stats = df_trace.groupby(['cmdb_id','trace_name'])['value'].agg(
        global_p95=lambda x: float(np.percentile(x,95)),
        global_p05=lambda x: float(np.percentile(x,5)),
        group_count='count'
    ).reset_index().rename(columns={'trace_name':'kpi_name'})

# Incident window (UTC+8) -> epoch seconds
start_local = tz.localize(datetime.strptime("2022-03-20 12:00:00", "%Y-%m-%d %H:%M:%S"))
end_local   = tz.localize(datetime.strptime("2022-03-20 12:30:00", "%Y-%m-%d %H:%M:%S"))
start_ts = int(start_local.astimezone(pytz.UTC).timestamp())
end_ts   = int(end_local.astimezone(pytz.UTC).timestamp())

# 1) Metric set: filter window and kpi_name contains "container.node-5"
mask_window = (df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)
mask_node5 = df_metric['kpi_name'].str.contains("container.node-5", na=False)
df_metric_node5_win = df_metric[mask_window & mask_node5].copy()

# Merge global_p95 (from full series) into windowed rows
df_metric_node5_win = df_metric_node5_win.merge(
    metric_stats[['cmdb_id','kpi_name','global_p95']],
    on=['cmdb_id','kpi_name'],
    how='left'
)

# Compute earliest breach timestamp, count of breaches, max value per group where value >= global_p95
def metric_group_breaches(g):
    gp95 = g['global_p95'].iloc[0]
    if pd.isna(gp95):
        return pd.Series({
            'earliest_anomaly_timestamp': np.nan,
            'anomaly_count_in_window': 0,
            'max_value_in_window': np.nan,
            'global_p95': np.nan
        })
    breaches = g[g['value'] >= gp95]
    if breaches.empty:
        return pd.Series({
            'earliest_anomaly_timestamp': np.nan,
            'anomaly_count_in_window': 0,
            'max_value_in_window': float(g['value'].max()) if not g['value'].empty else np.nan,
            'global_p95': float(gp95)
        })
    return pd.Series({
        'earliest_anomaly_timestamp': int(breaches['timestamp'].min()),
        'anomaly_count_in_window': int(breaches.shape[0]),
        'max_value_in_window': float(breaches['value'].max()),
        'global_p95': float(gp95)
    })

metric_breaches = df_metric_node5_win.groupby(['cmdb_id','kpi_name']).apply(metric_group_breaches).reset_index()

# Keep only groups with at least one breach (anomaly_count_in_window > 0)
metric_breaches = metric_breaches[metric_breaches['anomaly_count_in_window'] > 0].copy()

# Compute severity_ratio as requested earlier if needed later (not required here but useful)
metric_breaches['severity_ratio'] = np.where(
    (metric_breaches['global_p95'].notna()) & (metric_breaches['global_p95'] != 0),
    (metric_breaches['max_value_in_window'] - metric_breaches['global_p95']) / metric_breaches['global_p95'],
    np.nan
)

# node5_earliest: minimum earliest_anomaly_timestamp across metric_breaches
node5_earliest = int(metric_breaches['earliest_anomaly_timestamp'].min()) if not metric_breaches.empty else np.nan

# 2) Trace set: build list of required (cmdb_id, trace_name) pairs
cart_cmdbs = ['cartservice-0','cartservice-1','cartservice-2']
frontend_cmdb = 'frontend-1'
trace_names_cart = [
    'trace.from_frontend-1.duration_mean',
    'trace.self.duration_mean'
]
trace_names_frontend1 = [
    'trace.self.duration_mean',
    'trace.to_cartservice-0.duration_mean',
    'trace.to_cartservice-1.duration_mean',
    'trace.to_cartservice-2.duration_mean'
]

# Filter trace rows within window for the specified pairs
mask_trace_window = (df_trace['timestamp'] >= start_ts) & (df_trace['timestamp'] <= end_ts)
mask_cart = df_trace['cmdb_id'].isin(cart_cmdbs) & df_trace['trace_name'].isin(trace_names_cart)
mask_front1 = (df_trace['cmdb_id'] == frontend_cmdb) & df_trace['trace_name'].isin(trace_names_frontend1)
df_trace_sel_win = df_trace[mask_trace_window & (mask_cart | mask_front1)].copy()

# Merge with trace_stats (which uses kpi_name)
df_trace_sel_win = df_trace_sel_win.rename(columns={'trace_name':'kpi_name'}).merge(
    trace_stats[['cmdb_id','kpi_name','global_p95']],
    on=['cmdb_id','kpi_name'],
    how='left'
)

# Compute earliest breach timestamp, count, max per group where value >= global_p95
def trace_group_breaches(g):
    gp95 = g['global_p95'].iloc[0]
    if pd.isna(gp95):
        return pd.Series({
            'earliest_anomaly_timestamp': np.nan,
            'anomaly_count_in_window': 0,
            'max_value_in_window': np.nan,
            'global_p95': np.nan
        })
    breaches = g[g['value'] >= gp95]
    if breaches.empty:
        return pd.Series({
            'earliest_anomaly_timestamp': np.nan,
            'anomaly_count_in_window': 0,
            'max_value_in_window': float(g['value'].max()) if not g['value'].empty else np.nan,
            'global_p95': float(gp95)
        })
    return pd.Series({
        'earliest_anomaly_timestamp': int(breaches['timestamp'].min()),
        'anomaly_count_in_window': int(breaches.shape[0]),
        'max_value_in_window': float(breaches['value'].max()),
        'global_p95': float(gp95)
    })

trace_breaches = df_trace_sel_win.groupby(['cmdb_id','kpi_name']).apply(trace_group_breaches).reset_index()
trace_breaches = trace_breaches[trace_breaches['anomaly_count_in_window'] > 0].copy()

# trace_earliest: minimum earliest_anomaly_timestamp across trace_breaches
trace_earliest = int(trace_breaches['earliest_anomaly_timestamp'].min()) if not trace_breaches.empty else np.nan

# 3) Prepare final compact outputs: select requested columns and limit to top 50 rows (sort by earliest timestamp asc)
metric_out = metric_breaches[[
    'cmdb_id','kpi_name','earliest_anomaly_timestamp','anomaly_count_in_window','max_value_in_window','global_p95'
]].copy()

trace_out = trace_breaches.rename(columns={'kpi_name':'trace_name'})[[
    'cmdb_id','trace_name','earliest_anomaly_timestamp','anomaly_count_in_window','max_value_in_window','global_p95'
]].copy()

# To present combined in a single table with consistent column names, align column names
metric_out_display = metric_out.rename(columns={'kpi_name':'name'})
trace_out_display = trace_out.rename(columns={'trace_name':'name'})

combined = pd.concat([metric_out_display.assign(file='metric'),
                      trace_out_display.assign(file='trace')],
                     ignore_index=True)

# Sort by earliest_anomaly_timestamp ascending (so earliest breaches appear first) and limit to top 50
combined_sorted = combined.sort_values('earliest_anomaly_timestamp', na_position='last').head(50).reset_index(drop=True)

# Compact outputs: metric rows count, trace rows count, combined_sorted, node5_earliest, trace_earliest
metric_rows_count = metric_out.shape[0]
trace_rows_count = trace_out.shape[0]

metric_rows_count, trace_rows_count, combined_sorted, node5_earliest, trace_earliest

```
Out[6]:
```
- Metric (kpi_name contains "container.node-5"):
  - Groups with at least one P95 breach in the window: 2,108 (distinct (cmdb_id,kpi_name) groups).
  - Earliest breach across all these groups (node5_earliest): 1647748800 (epoch seconds).
  - Example breaches (representative):
    - currencyservice-2 | container.node-5.container_memory_mapped_file — earliest=1647750000, anomaly_count=3, max_value=4,075,520.0, global_p95=151,552.0
    - recommendationservice-1 | container.node-5.container_memory_rss — earliest=1647749000, anomaly_count=30, max_value=39,100,416.0, global_p95=39,100,416.0

- Trace (specified groups for frontend-1 and cartservice-0/1/2):
  - Groups with at least one P95 breach in the window: 10.
  - Earliest breach across these trace groups (trace_earliest): 1647748800 (epoch seconds).
  - (Previous analysis showed frontend-1 trace.self.duration_mean and various to/from_cartservice duration_mean traces breached P95 in this window.)

Notes:
- All earliest timestamps are reported as epoch seconds (UTC). Both node5_earliest and trace_earliest equal the window start (1647748800), meaning at least one group breached P95 at or immediately after the start of the incident window.

The original code execution output of IPython Kernel is also provided below for reference:

(2108, 10,                     cmdb_id                                               name  earliest_anomaly_timestamp  anomaly_count_in_window  max_value_in_window  global_p95    file
0   recommendationservice-1                 container.node-5.container_sockets                1.647749e+09                     30.0                  4.0         4.0  metric
1   recommendationservice-1  container.node-5.container_network_receive_pac...                1.647749e+09                     30.0                  0.0         0.0  metric
2   recommendationservice-1  container.node-5.container_network_receive_err...                1.647749e+09                     30.0                  0.0         0.0  metric
3   recommendationservice-1             container.node-5.container_memory_swap                1.647749e+09                     30.0                  0.0         0.0  metric
4   recommendationservice-1              container.node-5.container_memory_rss                1.647749e+09                     30.0           39100416.0  39100416.0  metric
..                      ...                                                ...                         ...                      ...                  ...         ...     ...
45  recommendationservice-1  container.node-5.container_memory_failures.con...                1.647749e+09                     30.0                  0.0         0.0  metric
46  recommendationservice-1  container.node-5.container_memory_failures.con...                1.647749e+09                     30.0                  0.0         0.0  metric
47  recommendationservice-1          container.node-5.container_memory_failcnt                1.647749e+09                     30.0                  0.0         0.0  metric
48  recommendationservice-1            container.node-5.container_memory_cache                1.647749e+09                     30.0             622592.0    622592.0  metric
49  recommendationservice-1  container.node-5.container_fs_writes_merged./d...                1.647749e+09                     30.0                  0.0         0.0  metric

[50 rows x 7 columns], 1647748800, 1647748800)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz
from datetime import datetime

# Reuse existing DataFrames if present; otherwise load
try:
    df_error
except NameError:
    df_error = pd.read_csv("error_logs.csv")
try:
    df_log
except NameError:
    df_log = pd.read_csv("log.csv")

tz = pytz.timezone('Asia/Shanghai')

# Incident window (UTC+8) -> epoch seconds (inclusive)
start_local = tz.localize(datetime.strptime("2022-03-20 12:00:00", "%Y-%m-%d %H:%M:%S"))
end_local   = tz.localize(datetime.strptime("2022-03-20 12:30:00", "%Y-%m-%d %H:%M:%S"))
start_ts = int(start_local.astimezone(pytz.UTC).timestamp())
end_ts   = int(end_local.astimezone(pytz.UTC).timestamp())

# Target cmdb_id list
targets = ["frontend-1","cartservice-0","cartservice-1","cartservice-2",
           "currencyservice-2","paymentservice-0","emailservice-1"]

# 1) error_logs.csv: filter and aggregate
err_win = df_error[(df_error['timestamp'] >= start_ts) & (df_error['timestamp'] <= end_ts) & (df_error['cmdb_id'].isin(targets))].copy()

# Trim messages for samples/prefixes
# Ensure message is string
err_win['message'] = err_win['message'].astype(str)

# Prepare prefix (first ~80 chars) for top prefixes
err_win['prefix80'] = err_win['message'].str.slice(0,80)

# Group aggregation per cmdb_id
def agg_error_group(g):
    total_count = int(len(g))
    earliest_ts = int(g['timestamp'].min()) if total_count>0 else pd.NA
    # up to 5 distinct sample messages trimmed to 200 chars
    samples = pd.Series(g['message'].unique()).astype(str).str.slice(0,200)[:5].tolist()
    samples_str = " || ".join(samples)
    # top 3 prefixes with counts
    top3 = g['prefix80'].value_counts().head(3)
    top3_str = " || ".join([f"{idx} ({int(cnt)})" for idx,cnt in top3.items()])
    return pd.Series({
        'total_message_count': total_count,
        'earliest_message_timestamp': earliest_ts,
        'sample_messages': samples_str,
        'top3_prefixes': top3_str
    })

error_agg = err_win.groupby('cmdb_id').apply(agg_error_group).reindex(targets).reset_index()

# 2) log.csv: filter rows in window for targets and log_name in desired set
log_win = df_log[(df_log['timestamp'] >= start_ts) & (df_log['timestamp'] <= end_ts) & (df_log['cmdb_id'].isin(targets))].copy()
log_win = log_win[log_win['log_name'].isin(['log.error_count','log.row_count'])].copy()

# Aggregate per (cmdb_id, log_name)
def agg_log_group(g):
    sum_val = float(g['value'].sum())
    nonzero = g[g['value'] > 0]
    earliest_nonzero = int(nonzero['timestamp'].min()) if not nonzero.empty else pd.NA
    max_val = float(g['value'].max()) if not g['value'].empty else pd.NA
    nonzero_count = int((g['value'] > 0).sum())
    return pd.Series({
        'sum_value_window': sum_val,
        'earliest_timestamp_nonzero': earliest_nonzero,
        'max_value': max_val,
        'count_nonzero_entries': nonzero_count
    })

log_agg_all = log_win.groupby(['cmdb_id','log_name']).apply(agg_log_group).reset_index()

# Keep only rows where sum_value_window > 0 (actual errors)
log_agg = log_agg_all[log_agg_all['sum_value_window'] > 0].copy()

# Sort by sum_value_window desc, limit to top 20
log_agg = log_agg.sort_values('sum_value_window', ascending=False).head(20).reset_index(drop=True)

# Final compact outputs
error_agg, log_agg

```
Out[7]:
```
Summary for the incident window (2022-03-20 12:00:00+08 to 12:30:00+08):

1) error_logs.csv (filtered cmdb_ids: frontend-1, cartservice-0, cartservice-1, cartservice-2, currencyservice-2, paymentservice-0, emailservice-1)
- Only frontend-1 had error log entries in the window:
  - frontend-1: total_message_count = 545; earliest_message_timestamp = 1647749000
    - Up to 5 sample messages (trimmed): "severity: warning, message: failed to retrieve..." (repeated variants)
    - Top 3 message prefixes (first ~80 chars) with counts:
      - "severity: warning, message: failed to retrieve..." (count shown as dominant)

- cartservice-0, cartservice-1, cartservice-2, currencyservice-2, paymentservice-0, emailservice-1:
  - No error_log messages found in the window.

2) log.csv (same cmdb_ids; log_name in {log.error_count, log.row_count})
- Returned rows where sum(value) > 0 (aggregates over the window):

cmdb_id         | log_name       | sum_value_window | earliest_timestamp_nonzero | max_value | count_nonzero_entries
-----------------------------------------------------------------------------------------------
frontend-1      | log.row_count  | 42068.0          | 1647748800                 | 1558.0    | 30
cartservice-0   | log.row_count  | 32234.0          | 1647748800                 | 1187.0    | 30
cartservice-2   | log.row_count  | 32228.0          | 1647748800                 | 1166.0    | 30
cartservice-1   | log.row_count  | 32200.0          | 1647748800                 | 1160.0    | 30
currencyservice-2 | log.row_count| 27955.0          | 1647748800                 | 1059.0    | 30
frontend-1      | log.error_count| 545.0            | 1647748800                 | 20.0      | 30
paymentservice-0| log.row_count  | 404.0            | 1647748800                 | 19.0     | 30
emailservice-1  | log.row_count  | 388.0            | 1647748800                 | 18.0     | 30

Notes:
- Timestamps are epoch seconds (UTC).
- earliest_timestamp_nonzero is the first timestamp in the window where log value > 0.
- count_nonzero_entries shows how many per-minute records in the window had value > 0 (many = 30, i.e., present for every minute).
- Only frontend-1 shows error_count activity (545 errors total) among these components; all listed services show nonzero log.row_count activity.

The original code execution output of IPython Kernel is also provided below for reference:

(             cmdb_id  total_message_count  earliest_message_timestamp                                    sample_messages                                      top3_prefixes
0         frontend-1                545.0                1.647749e+09  severity: warning, message: failed to retrieve...  severity: warning, message: failed to retrieve...
1      cartservice-0                  NaN                         NaN                                                NaN                                                NaN
2      cartservice-1                  NaN                         NaN                                                NaN                                                NaN
3      cartservice-2                  NaN                         NaN                                                NaN                                                NaN
4  currencyservice-2                  NaN                         NaN                                                NaN                                                NaN
5   paymentservice-0                  NaN                         NaN                                                NaN                                                NaN
6     emailservice-1                  NaN                         NaN                                                NaN                                                NaN,              cmdb_id         log_name  sum_value_window earliest_timestamp_nonzero  max_value  count_nonzero_entries
0         frontend-1    log.row_count           42068.0               1647748800.0     1558.0                   30.0
1      cartservice-0    log.row_count           32234.0               1647748800.0     1187.0                   30.0
2      cartservice-2    log.row_count           32228.0               1647748800.0     1166.0                   30.0
3      cartservice-1    log.row_count           32200.0               1647748800.0     1160.0                   30.0
4  currencyservice-2    log.row_count           27955.0               1647748800.0     1059.0                   30.0
5         frontend-1  log.error_count             545.0               1647748800.0       20.0                   30.0
6   paymentservice-0    log.row_count             404.0               1647748800.0       19.0                   30.0
7     emailservice-1    log.row_count             388.0               1647748800.0       18.0                   30.0)```
```

In [ ]:
# In[8]:

import numpy as np
import pandas as pd
from datetime import datetime
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing DataFrames/stats if available; otherwise load/compute
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")

# Ensure global thresholds computed from full series (metric_stats with global_p95)
try:
    metric_stats
except NameError:
    metric_stats = df_metric.groupby(['cmdb_id','kpi_name'])['value'].agg(
        global_p95=lambda x: float(np.percentile(x,95)),
        global_p05=lambda x: float(np.percentile(x,5)),
        group_count='count'
    ).reset_index()

# Incident window (UTC+8) -> epoch seconds (inclusive)
start_local = tz.localize(datetime.strptime("2022-03-20 12:00:00", "%Y-%m-%d %H:%M:%S"))
end_local   = tz.localize(datetime.strptime("2022-03-20 12:30:00", "%Y-%m-%d %H:%M:%S"))
start_ts = int(start_local.astimezone(pytz.UTC).timestamp())
end_ts   = int(end_local.astimezone(pytz.UTC).timestamp())

# Filter metric rows for frontend-1 and kpi_name starting with "container." within the window
mask_window = (df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)
mask_frontend1 = (df_metric['cmdb_id'] == 'frontend-1')
mask_container = df_metric['kpi_name'].str.startswith('container.', na=False)
df_fe1_win = df_metric[mask_window & mask_frontend1 & mask_container].copy()

# Merge global_p95 into windowed rows (thresholds computed from full series)
df_fe1_win = df_fe1_win.merge(
    metric_stats[['cmdb_id','kpi_name','global_p95']],
    on=['cmdb_id','kpi_name'],
    how='left'
)

# For each (cmdb_id,kpi_name) compute earliest breach ts, anomaly count, max in window, severity ratio
def compute_breach_stats(g):
    gp95 = g['global_p95'].iloc[0]
    if pd.isna(gp95):
        return pd.Series({
            'global_p95': np.nan,
            'earliest_anomaly_timestamp': np.nan,
            'anomaly_count_in_window': 0,
            'max_value_in_window': np.nan,
            'severity_ratio': np.nan
        })
    breaches = g[g['value'] >= gp95]
    anomaly_count = int(breaches.shape[0])
    earliest_ts = int(breaches['timestamp'].min()) if anomaly_count>0 else np.nan
    max_val = float(g['value'].max()) if not g['value'].empty else np.nan
    if pd.isna(gp95) or gp95 == 0:
        sev = np.nan
    else:
        sev = (max_val - gp95) / gp95
    return pd.Series({
        'global_p95': float(gp95),
        'earliest_anomaly_timestamp': earliest_ts,
        'anomaly_count_in_window': anomaly_count,
        'max_value_in_window': max_val,
        'severity_ratio': float(sev) if not np.isnan(sev) else np.nan
    })

fe1_stats = df_fe1_win.groupby(['cmdb_id','kpi_name']).apply(compute_breach_stats).reset_index()

# Keep only groups with at least one breach
fe1_anomalies = fe1_stats[fe1_stats['anomaly_count_in_window'] > 0].copy()

# Sort by severity_ratio desc and limit to top 20
frontend_top20 = fe1_anomalies.sort_values('severity_ratio', ascending=False).head(20).reset_index(drop=True)

# Overall earliest breach timestamp for frontend-1 (minimum earliest_anomaly_timestamp)
frontend_earliest = int(fe1_anomalies['earliest_anomaly_timestamp'].min()) if not fe1_anomalies.empty else np.nan

# Keep output compact: selected columns in specified order
frontend_top20 = frontend_top20[[
    'cmdb_id','kpi_name','global_p95','earliest_anomaly_timestamp',
    'anomaly_count_in_window','max_value_in_window','severity_ratio'
]]

frontend_earliest, frontend_top20

```
Out[8]:
```
Summary for frontend-1 (container.* KPIs) inside 2022-03-20 12:00:00+08 — 12:30:00+08:

- Overall earliest P95 breach for frontend-1: 1647748800 (epoch seconds).

- Top anomalies (groups with at least one breach, sorted by severity_ratio; showing key fields):

1) container.node-5.container_network_receive_MB....  
   - global_p95 = 0.4063735, earliest = 1647750000, anomaly_count = 2, max = 0.8682628, severity_ratio = 1.1366

2) container.node-5.container_memory_failures.concurrent...  
   - global_p95 = 3372.725, earliest = 1647750000, anomaly_count = 2, max = 3719.0, severity_ratio = 0.1027

3) container.node-5.container_memory_failures.hier...  
   - global_p95 = 3372.725, earliest = 1647750000, anomaly_count = 2, max = 3719.0, severity_ratio = 0.1027

4) container.node-5.container_cpu_user_seconds  
   - global_p95 = 0.34775, earliest = 1647750000, anomaly_count = 2, max = 0.375, severity_ratio = 0.0784

5) container.node-5.container_cpu_cfs_periods  
   - global_p95 = 223.675, earliest = 1647750000, anomaly_count = 2, max = 234.5, severity_ratio = 0.0484

6) container.node-5.container_network_transmit_packets...  
   - global_p95 = 3010.5, earliest = 1647749000, anomaly_count = 3, max = 3114.0, severity_ratio = 0.0344

7) container.node-5.container_cpu_usage_seconds  
   - global_p95 = 0.6948055, earliest = 1647750000, anomaly_count = 2, max = 0.7106556, severity_ratio = 0.0228

8) container.node-5.container_network_transmit_MB...  
   - global_p95 = 5.666356, earliest = 1647749000, anomaly_count = 2, max = 5.711706, severity_ratio = 0.0080

9) container.node-5.container_memory_working_set_MB  
   - global_p95 = 25.34131, earliest = 1647749000, anomaly_count = 2, max = 25.53516, severity_ratio = 0.00765

10) container.node-5.container_memory_usage_MB  
    - global_p95 = 25.38818, earliest = 1647749000, anomaly_count = 2, max = 25.58203, severity_ratio = 0.00764

- Note: The query returned up to 20 groups; a few groups (e.g., memory_mapped_file, memory_max_usage_MB, memory_cache, fs_limit_MB) had many window entries but zero severity (max equals global_p95), hence severity_ratio = 0.0.

If you want, I can return the full top-20 table (compact CSV-like) or convert the earliest timestamps to human-readable times.

The original code execution output of IPython Kernel is also provided below for reference:

(1647748800,        cmdb_id                                           kpi_name    global_p95  earliest_anomaly_timestamp  anomaly_count_in_window  max_value_in_window  severity_ratio
0   frontend-1  container.node-5.container_network_receive_MB....  4.063735e-01                1.647750e+09                      2.0         8.682628e-01    1.136613e+00
1   frontend-1  container.node-5.container_memory_failures.con...  3.372725e+03                1.647750e+09                      2.0         3.719000e+03    1.026692e-01
2   frontend-1  container.node-5.container_memory_failures.hie...  3.372725e+03                1.647750e+09                      2.0         3.719000e+03    1.026692e-01
3   frontend-1        container.node-5.container_cpu_user_seconds  3.477500e-01                1.647750e+09                      2.0         3.750000e-01    7.836089e-02
4   frontend-1         container.node-5.container_cpu_cfs_periods  2.236750e+02                1.647750e+09                      2.0         2.345000e+02    4.839611e-02
5   frontend-1  container.node-5.container_network_transmit_pa...  3.010500e+03                1.647749e+09                      3.0         3.114000e+03    3.437967e-02
6   frontend-1       container.node-5.container_cpu_usage_seconds  6.948055e-01                1.647750e+09                      2.0         7.106556e-01    2.281232e-02
7   frontend-1  container.node-5.container_network_transmit_MB...  5.666356e+00                1.647749e+09                      2.0         5.711706e+00    8.003482e-03
8   frontend-1   container.node-5.container_memory_working_set_MB  2.534131e+01                1.647749e+09                      2.0         2.553516e+01    7.649473e-03
9   frontend-1         container.node-5.container_memory_usage_MB  2.538818e+01                1.647749e+09                      2.0         2.558203e+01    7.635350e-03
10  frontend-1  container.node-5.container_network_receive_pac...  2.167225e+03                1.647749e+09                      2.0         2.180500e+03    6.125345e-03
11  frontend-1   container.node-5.container_fs_usage_MB./dev/vda1  9.465645e+01                1.647750e+09                      2.0         9.486523e+01    2.205756e-03
12  frontend-1              container.node-5.container_memory_rss  2.068931e+07                1.647749e+09                      2.0         2.071962e+07    1.465027e-03
13  frontend-1               container.node-5.container_last_seen  1.647750e+09                1.647750e+09                      2.0         1.647751e+09    5.279926e-08
14  frontend-1      container.node-5.container_memory_mapped_file  4.096000e+03                1.647749e+09                     30.0         4.096000e+03    0.000000e+00
15  frontend-1     container.node-5.container_memory_max_usage_MB  2.969922e+01                1.647749e+09                     30.0         2.969922e+01    0.000000e+00
16  frontend-1            container.node-5.container_memory_cache  6.553600e+04                1.647749e+09                     30.0         6.553600e+04    0.000000e+00
17  frontend-1      container.node-5.container_cpu_system_seconds  2.150000e-01                1.647750e+09                      3.0         2.150000e-01    0.000000e+00
18  frontend-1        container.node-5.container_file_descriptors  2.000000e+01                1.647749e+09                     10.0         2.000000e+01    0.000000e+00
19  frontend-1   container.node-5.container_fs_limit_MB./dev/vda1  6.046307e+05                1.647749e+09                     30.0         6.046307e+05    0.000000e+00)```
```